In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import euclidean_distances
import os
from pathlib import Path
import json

d:\Users\Daniel Hamill\Documents\Projects\discord-llm-agent\history-bot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
data = pd.read_csv('exported_messages.csv', names=["id", "user", "content", "timestamp"])

In [9]:
# Load the pre-trained sentence transformer model
model_name =  "BAAI/bge-base-en-v1.5"
model_path = Path.home() / Path("models", model_name)

In [13]:
model = SentenceTransformer(model_name)

d:\Users\Daniel Hamill\Documents\Projects\discord-llm-agent\history-bot\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Daniel Hamill\.cache\huggingface\hub\models--BAAI--bge-base-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4739.

In [19]:
data["content"][0]

'Reminder for anyone who wants to come there’s an art sale at lavender bookshop on the 11th. I will be there right at noon because I have to leave to go to work later'

In [20]:
data["content"][1]

'The only issue is my parents are...well...my parents. And only a few of yall have met them before'

In [26]:
data["content"][8]

'If anyone wants to come'

In [27]:
len(data)

8482

In [30]:
# Embed all messages and the query
messages = data["content"].dropna().tolist()
message_embeddings = model.encode(messages, show_progress_bar=True)


In [34]:
# Example: find the most similar messages to a query
query = "I'm planning on hosting/throwing an event, party or hangout if anyone want to hang out in the near future."
query_embedding = model.encode([query])

# Cosine similarity: higher score = more similar
cos_scores = cosine_similarity(query_embedding, message_embeddings)[0]

# Euclidean distance: lower score = more similar
euc_distances = euclidean_distances(query_embedding, message_embeddings)[0]

# Top-5 most similar messages by cosine similarity
top_idx = np.argsort(cos_scores)[::-1][:15]
print(f"Query: {query!r}\n")
print("Top 5 similar messages (cosine similarity):")
for rank, i in enumerate(top_idx, 1):
    print(f"  {rank}. [cos={cos_scores[i]:.4f}, euc={euc_distances[i]:.4f}] {messages[i]!r}")

Query: "I'm planning on hosting/throwing an event, party or hangout if anyone want to hang out in the near future."

Top 5 similar messages (cosine similarity):
  1. [cos=0.8325, euc=0.5788] 'I’m probably gonna host a new years party'
  2. [cos=0.8167, euc=0.6055] 'I have ideas and can host'
  3. [cos=0.7996, euc=0.6330] 'Is anyone planning on hosting a Fourth of July party this year?'
  4. [cos=0.7895, euc=0.6488] "I'm also happy to host at my place, we could hang out and talk afterwards and I could maybe even make a meal for everyone :)"
  5. [cos=0.7879, euc=0.6514] 'I should prolly start planning a new year party'
  6. [cos=0.7845, euc=0.6565] 'Btw, unless there are objections, i would like to host New Years party this year'
  7. [cos=0.7840, euc=0.6573] "I'll also throw this out there, I'm hosting"
  8. [cos=0.7797, euc=0.6638] 'I am absolutely down to host it'
  9. [cos=0.7791, euc=0.6647] 'If any of y’all are interested Nick + I might be throwing a back to school party thingmaji

In [39]:
message_embeddings

array([[-0.02591967, -0.02137849,  0.05155126, ...,  0.02903723,
         0.03371389, -0.01352069],
       [-0.04125561,  0.01487034, -0.00428558, ...,  0.01319505,
         0.04340078,  0.00645474],
       [ 0.01309068,  0.00802773, -0.00572118, ...,  0.00927786,
         0.00216123,  0.0045518 ],
       ...,
       [ 0.01123792,  0.02326594,  0.06034489, ..., -0.05415989,
         0.00349689,  0.02409916],
       [ 0.01204387,  0.00953662, -0.00471921, ..., -0.010098  ,
         0.00660921, -0.00461826],
       [ 0.05032211, -0.02273464, -0.00441087, ..., -0.00159333,
         0.00186223, -0.01465196]], shape=(8228, 768), dtype=float32)

In [37]:
data.shape

(8482, 4)

In [41]:
clean_data = data.dropna(subset=["content"])

In [42]:
clean_data.shape

(8228, 4)

In [ ]:
clean_data["embedding"] = [json.dumps(e.tolist()) for e in message_embeddings]

clean_data.to_csv('exported_messages.csv', index=False)


In [2]:
def load_data(csv_path: str = 'exported_messages.csv'):
    import json
    import numpy as np
    import pandas as pd

    df = pd.read_csv(csv_path)
    embeddings = np.array(df["embedding"].apply(json.loads).tolist())
    return df, embeddings

df, embeddings = load_data()


In [4]:
embeddings.shape

(8228, 768)